# 02 · Memory — 03 semantic recall (finding the right memory, not the recent one)

**The lesson this notebook exists to land: memory and retrieval are the same mechanism pointed at different corpora.** `01-tools/03-embed` and `01-tools/04-retrieve` embed *documents* and rank them against a query. Point the identical machinery at the agent's own past — its episodes, its previous decisions — and it is a memory system. There is no second technology. Nothing in this notebook is new machinery; it is `hash_embed` from `01-tools/03-embed/01-offline-embeddings.ipynb`, reused verbatim, with the corpus swapped.

`02-long-term.ipynb` ended on a real failure: `find_similar_deductions` is a substring match, so the same mistake described in different words finds nothing and gets graded cold. This notebook replaces `q in hay` with vector similarity.

**The honest part, up front.** The embedding used here is a deterministic hash of whitespace tokens. It is not semantically meaningful and this notebook does not claim it is. What it can guarantee is narrow and checked: a memory recalls *itself* at a similarity of exactly 1.0, and a query sharing tokens with a memory outranks one that does not. What it cannot do is also demonstrated, with real output — Step 9 shows a true paraphrase scoring **0.0** against the memory it paraphrases while scoring higher against an unrelated one, i.e. ranking the wrong memory first. A real embedding model is the thing that makes semantic recall work; the hash proves the plumbing, not the semantics.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `hash_embed` | Deterministic, offline embedding — SHA-256 over whitespace tokens into a fixed-size L2-normalised vector. | `hash_embed("no units given", dim=384)` |
| `cosine` | Similarity between two unit vectors; a plain dot product, since both are normalised. | `cosine(a, b)` → `1.0` |
| `MemoryStore` | An episodic store: append a memory, recall by recency or by vector similarity. | `store.recall_semantic("wrong sign", k=3)` |
| `recall_recency` / `recall_semantic` | The two sort orders this whole notebook is about. | `store.recall_recency(3)` vs `store.recall_semantic(q, 3)` |
| `embed_real` | The same store on OpenAI embeddings if `OPENAI_API_KEY` is set — the path that makes recall actually semantic. | `embed_real(texts)` |

## Step 1 — bootstrap the repo path and confirm the environment

`show_environment()` is what tells a reader which of the two paths below they are on: the hash path (always), and the real-embedding comparison in Step 11 (only with `OPENAI_API_KEY`).

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

# bootstrap() above already put the repo root on sys.path and loaded .env;
# `_root` is that same path, reused here rather than resolved a second time.
repo_root = _root
env = nbio.show_environment()

## Step 2 — `hash_embed`, reused from the embed stage

Copied from `01-modules/01-tools/03-embed/01-offline-embeddings.ipynb` without modification. That it needs no modification is the point: the function does not know or care whether the strings it is given are document chunks or an agent's own past turns.

In [ ]:
import hashlib
import math


def hash_embed(text: str, dim: int = 384) -> list[float]:
    """Deterministic, offline embedding -- no model, no API key, no network."""
    vec = [0.0] * dim
    tokens = (text or "").lower().split()
    if not tokens:
        return vec
    for tok in tokens:
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        idx = h % dim
        sign = 1.0 if (h >> 8) & 1 else -1.0
        vec[idx] += sign
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def cosine(a: list[float], b: list[float]) -> float:
    """Both vectors are L2-normalised, so cosine similarity is the dot product."""
    return sum(x * y for x, y in zip(a, b))


DIM = 384
print(f"dim={len(hash_embed('a test memory', dim=DIM))}")

## Step 3 — the corpus is the agent's own past

Five episodes, in the shape `02-long-term.ipynb` wrote to disk: an error type, what it cost, and why. Synthetic, invented here. What matters is that these are *the agent's memories*, not documents — and that nothing below treats them differently from documents.

In [ ]:
EPISODES = [
    {"error_type": "missing units",
     "text": "student forgot the units on the final answer, deducted 2 marks",
     "points_deducted": 2.0, "session": 1},
    {"error_type": "sign error in integration",
     "text": "student used the wrong sign in the integration step, deducted 3 marks",
     "points_deducted": 3.0, "session": 1},
    {"error_type": "unlabelled free-body diagram",
     "text": "student left the free-body diagram unlabelled, deducted 1 mark",
     "points_deducted": 1.0, "session": 2},
    {"error_type": "arithmetic slip",
     "text": "student copied the formula correctly but arithmetic slipped, deducted 1 mark",
     "points_deducted": 1.0, "session": 3},
    {"error_type": "assumptions not stated",
     "text": "student did not state assumptions before applying the ideal gas law, deducted 2 marks",
     "points_deducted": 2.0, "session": 4},
]

nbio.table([(e["session"], e["error_type"], e["points_deducted"]) for e in EPISODES],
           ("session", "error type", "marks"))

## Step 4 — a store with two recall orders

`MemoryStore` holds the episodes and their vectors. `recall_recency` returns the most recent memories; `recall_semantic` returns the closest ones. The class is fifteen lines and holds no model — swapping `embed_fn` is the only thing that changes when a real embedding model is available.

In [ ]:
class MemoryStore:
    """Episodic memory with two recall orders: by recency, and by vector similarity."""

    def __init__(self, embed_fn=hash_embed):
        self.embed_fn = embed_fn
        self.records: list[dict] = []
        self.vectors: list[list[float]] = []

    def add(self, record: dict) -> None:
        self.records.append(record)
        self.vectors.append(self.embed_fn(record["text"]))

    def recall_recency(self, k: int = 3) -> list[dict]:
        return list(reversed(self.records[-k:]))

    def recall_semantic(self, query: str, k: int = 3) -> list[tuple[float, dict]]:
        qv = self.embed_fn(query)
        scored = [(cosine(qv, v), r) for v, r in zip(self.vectors, self.records)]
        scored.sort(key=lambda sr: -sr[0])
        return scored[:k]


store = MemoryStore()
for e in EPISODES:
    store.add(e)

print(f"{len(store.records)} memories stored, {len(store.vectors[0])}-dim vectors")
assert len(store.records) == len(store.vectors) == 5

## Step 5 — the guarantee this embedding *can* make: exact self-recall

A memory retrieved by its own text scores exactly 1.0, and ranks first. This holds because the embedding is deterministic and normalised — there is no model state, no sampling, and no approximation anywhere in the path. It is the one claim about this vector space that is airtight.

In [ ]:
target = EPISODES[1]
top = store.recall_semantic(target["text"], k=3)

for score, rec in top:
    print(f"  {score:+.4f}  {rec['error_type']}")
print()

best_score, best_rec = top[0]
assert best_rec["error_type"] == target["error_type"], "a memory must recall itself first"
assert abs(best_score - 1.0) < 1e-9, f"exact self-recall must score 1.0, got {best_score}"
print(f"exact self-recall: {best_score:.10f} — 1.0 to floating-point precision")

## Step 6 — recency and meaning are different sort orders

The query is about a sign error, recorded in session 1 — the *oldest* memory but one. Recency puts the last three sessions on top and never reaches it. Similarity puts it first.

This is the whole argument for semantic recall in one table: the right memory is usually not the recent one.

In [ ]:
query = "wrong sign"

by_recency = store.recall_recency(3)
by_meaning = store.recall_semantic(query, 3)

print(f"query: {query!r}\n")
nbio.table(
    [(i + 1, r["error_type"], f"session {r['session']}") for i, r in enumerate(by_recency)],
    ("#", "recall by recency", "when"),
)
print()
nbio.table(
    [(i + 1, r["error_type"], f"{s:+.4f}") for i, (s, r) in enumerate(by_meaning)],
    ("#", "recall by similarity", "score"),
)
print()

recency_types = [r["error_type"] for r in by_recency]
assert "sign error in integration" not in recency_types, "recency never reaches it"
assert by_meaning[0][1]["error_type"] == "sign error in integration", "similarity puts it first"
print("Recency misses the relevant memory entirely; similarity ranks it first.")

## Step 7 — what this fixes in `02-long-term.ipynb`

That notebook's `find_similar_deductions` is `q in hay`. A query phrased differently returns `[]`, the grader falls through to a cold default, and the consistency the memory exists to enforce quietly does not happen. Side by side on the same query, with both implementations in front of you.

In [ ]:
def substring_recall(query: str, episodes: list[dict]) -> list[dict]:
    """The Step-13 lookup from 02-long-term.ipynb, reproduced for comparison."""
    q = (query or "").lower()
    return [e for e in episodes if q in f"{e['error_type']} {e['text']}".lower()]


probe = "no units given"

sub = substring_recall(probe, EPISODES)
sem = store.recall_semantic(probe, k=1)

print(f"query: {probe!r}")
print(f"  substring match : {len(sub)} hit(s)")
print(f"  vector match    : {sem[0][1]['error_type']!r} at {sem[0][0]:+.4f}")
print()

assert sub == [], "the substring lookup finds nothing for this phrasing"
assert sem[0][1]["error_type"] == "missing units", "the vector lookup finds the right memory"
print("Same query, same corpus: one returns nothing, the other returns the right memory.")

## Step 8 — and be precise about *why* it worked

It worked because `units` is a literal token in both strings. This is token overlap, not meaning. Saying so now is what makes Step 9 a demonstration rather than a surprise.

In [ ]:
shared = set(probe.lower().split()) & set(EPISODES[0]["text"].lower().split())
print(f"query tokens  : {sorted(set(probe.lower().split()))}")
print(f"memory tokens : {sorted(set(EPISODES[0]['text'].lower().split()))}")
print(f"shared        : {sorted(shared)}")

assert shared, "the match in Step 7 rests entirely on this overlap"

## Step 9 — what a hash embedding cannot do, shown rather than asserted

A true paraphrase: *"omitted dimensional annotation from their result"* means precisely the same thing as *"forgot the units on the final answer"* and shares not one token with it.

Under a hash embedding it scores **0.0** against the memory it paraphrases — orthogonal, the same score an unrelated string gets — and it scores *higher* against a memory about the ideal gas law, which it has nothing to do with. The top-ranked result is simply wrong.

A hash has no notion of meaning. It cannot; there is no training, no corpus, nothing but SHA-256. This is the boundary of what the offline path proves.

In [ ]:
paraphrase = "omitted dimensional annotation from their result"

ranked = store.recall_semantic(paraphrase, k=5)
nbio.table([(f"{s:+.4f}", r["error_type"]) for s, r in ranked], ("score", "memory"))
print()

scores = {r["error_type"]: s for s, r in ranked}
true_match = scores["missing units"]
top_score, top_rec = ranked[0]

print(f"score against the memory it actually paraphrases : {true_match:+.4f}")
print(f"top-ranked memory                                : {top_rec['error_type']!r} at {top_score:+.4f}")
print()

assert true_match == 0.0, "no shared tokens means an orthogonal vector — exactly zero similarity"
assert top_rec["error_type"] != "missing units", "the wrong memory ranks first"
print("A perfect paraphrase scores zero. Semantic recall is not what this embedding does.")

## Step 10 — a second, quieter failure: shared boilerplate is the whole score

Every episode is phrased the same way — `student … deducted N marks`. A hash weights `student` and `deducted` exactly as heavily as `integration`, because it has no way to know one is informative and the others are not. So two memories about completely different mistakes score over 0.5 against each other, purely on boilerplate.

Strip the shared filler words and the similarity between those two unrelated memories goes to **0.0**. Every point of it was formatting. A real embedding model learns to downweight uninformative tokens as a consequence of training; nothing in a hash can.

In [ ]:
FILLER = {"student", "the", "on", "a", "in", "deducted", "marks", "mark", "did", "not", "but"}


def strip_filler(text: str) -> str:
    return " ".join(w for w in text.split() if w.lower() not in FILLER)


a, b = EPISODES[0]["text"], EPISODES[1]["text"]          # two unrelated errors

with_filler = cosine(hash_embed(a, DIM), hash_embed(b, DIM))
without_filler = cosine(hash_embed(strip_filler(a), DIM), hash_embed(strip_filler(b), DIM))

print(f"as written                : {with_filler:+.4f}")
print(f"  {strip_filler(a)!r}")
print(f"  {strip_filler(b)!r}")
print(f"filler words removed      : {without_filler:+.4f}")
print(f"attributable to boilerplate: {with_filler - without_filler:+.4f}")
print()

assert without_filler == 0.0, "with the shared phrasing gone, the two share nothing at all"
print("Two unrelated memories scored 0.52 against each other on shared phrasing alone.")

## Step 11 — the same store on a real embedding model, if a key is set

Nothing about `MemoryStore` changes — only `embed_fn`. That is the point of Step 4 being fifteen lines: the difference between a toy and a working semantic memory is which function computes the vectors, not the memory architecture around it.

With no key this prints that plainly and skips; the hash store above has already proven the wiring, and Step 9 has already said honestly what it cannot do. The real call runs under `nbio.cost_meter` with a 50-cent ceiling — six short strings on `text-embedding-3-small` cost a small fraction of a cent.

In [ ]:
import os

EMBED_MODEL = "text-embedding-3-small"
has_key = bool(env.get("OPENAI_API_KEY"))
_usage = {"prompt_tokens": 0}


def embed_real(texts: list[str]) -> list[list[float]]:
    """Embed with OpenAI, recording token usage for the cost meter."""
    from openai import OpenAI

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    _usage["prompt_tokens"] += getattr(resp.usage, "prompt_tokens", 0)
    return [d.embedding for d in resp.data]


with nbio.cost_meter(budget_usd=0.50) as meter:
    if not has_key:
        print(
            "OPENAI_API_KEY not set, skipping — deterministic stand-in instead.\n"
            "The hash store above ran the full recall loop with no account anywhere.\n"
            "What it cannot do is Step 9: rank a paraphrase with no shared tokens.\n"
            "Set OPENAI_API_KEY and re-run this cell to see that same paraphrase\n"
            "score against the memory it paraphrases under a model that was trained\n"
            "on meaning rather than on hashing."
        )
    else:
        texts = [e["text"] for e in EPISODES] + [paraphrase]
        vecs = embed_real(texts)
        meter.record(EMBED_MODEL, _usage["prompt_tokens"], 0)

        real_store = MemoryStore(embed_fn=lambda t, _cache=dict(zip(texts, vecs)): _cache[t])
        for e in EPISODES:
            real_store.add(e)

        qv = vecs[-1]
        ranked_real = sorted(
            ((cosine(qv, v), r) for v, r in zip(real_store.vectors, real_store.records)),
            key=lambda sr: -sr[0],
        )
        nbio.table([(f"{s:+.4f}", r["error_type"]) for s, r in ranked_real], ("score", "memory"))
        print()
        print(f"hash embedding ranked 'missing units' at {true_match:+.4f} (rank "
              f"{[r['error_type'] for _, r in ranked].index('missing units') + 1} of 5)")
        print(f"real embedding ranks it at "
              f"{dict((r['error_type'], s) for s, r in ranked_real)['missing units']:+.4f} (rank "
              f"{[r['error_type'] for _, r in ranked_real].index('missing units') + 1} of 5)")

print()
print(meter.report())

## Step 12 — the claim, restated as code

One function, two corpora. `01-tools/03-embed` embeds document chunks; this notebook embeds the agent's own past. The embedding call is byte-identical and the ranking loop is the same three lines. If you already know how retrieval works, you already know how semantic memory works.

In [ ]:
documents = [
    "A vector embedding maps a passage of text onto a point in a high-dimensional space.",
    "Retrieval-augmented generation grounds an answer in retrieved passages.",
    "An index manifest records the model and dimension used to build a vector store.",
]
memories = [e["text"] for e in EPISODES]


def rank(query: str, corpus: list[str]) -> list[tuple[float, str]]:
    """The same three lines, whichever corpus is passed in."""
    qv = hash_embed(query, DIM)
    scored = [(cosine(qv, hash_embed(c, DIM)), c) for c in corpus]
    return sorted(scored, key=lambda sc: -sc[0])


print("corpus = documents      query = 'vector embedding'")
for s, c in rank("vector embedding", documents)[:1]:
    print(f"  {s:+.4f}  {c[:66]}")
print()
print("corpus = the agent's own past   query = 'wrong sign'")
for s, c in rank("wrong sign", memories)[:1]:
    print(f"  {s:+.4f}  {c[:66]}")
print()

assert "embedding" in rank("vector embedding", documents)[0][1]
assert "wrong sign" in rank("wrong sign", memories)[0][1]
print("Same embed call, same ranking loop, same guarantees. Only the corpus differs.")

## Where this fits

This closes the stage. `01-short-term.ipynb` shows memory inside one conversation and what a finite window costs; `02-long-term.ipynb` writes a decision past the end of a process; this notebook makes finding the right one tractable when there are more of them than a prompt can hold.

The dependency it makes explicit runs backwards through the repo: a memory system is a retrieval system, so everything `01-tools/03-embed` and `01-tools/04-retrieve` establish — dimension discipline, index manifests, ranking, deduplication — applies here unchanged.

## What did not come across

- **Semantics.** Said plainly: the offline path in this notebook does not do semantic recall. It does token-overlap recall and scores a perfect paraphrase at 0.0 (Step 9). A real embedding model is the component that makes the idea work; Step 11 is where it plugs in, and it needs a key.
- **A persistent vector index.** `MemoryStore` is a Python list and a linear scan, rebuilt on every kernel start. Real episodic memory at scale needs a store that survives a restart and does not compare against every memory on every query — `01-tools/03-embed/05-local-vector-store.ipynb` has the offline Chroma version, and combining it with `02-long-term.ipynb`'s file store is the obvious next step this stage does not take.
- **Writing memories back.** Every episode here is handed in ready-made. Deciding *what is worth remembering* out of a conversation — summarisation, salience, deduplication against what is already stored — is the hardest unsolved part of agent memory and is not attempted anywhere in this stage.
- **Forgetting.** No decay, no relevance-weighted eviction, no contradiction handling. `02-long-term.ipynb` Step 12 shows the cap that TerrierTA actually ships; nothing better is implemented here.
- **Hybrid recall.** A real system would blend similarity with recency and with `count`, rather than choosing one sort order. Step 6 contrasts the two precisely because the blend is where the engineering actually lives.